In [0]:
%sql
CREATE TABLE IF NOT EXISTS capstone.metadata.pipeline_log (
    pipeline_name STRING,
    layer STRING,
    run_time TIMESTAMP,
    status STRING,
    records_processed BIGINT
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS capstone.metadata.watermark (
    table_name STRING,
    last_loaded_date DATE
);


In [0]:
%sql
INSERT INTO capstone.metadata.watermark 
VALUES ('admissions', '2020-01-01');

num_affected_rows,num_inserted_rows
1,1


In [0]:
# =========================================
# GET WATERMARK
# =========================================

watermark = spark.table("capstone.metadata.watermark") \
    .filter("table_name = 'admissions'") \
    .select("last_loaded_date") \
    .collect()[0][0]

print("Last Watermark:", watermark)

Last Watermark: 2024-06-18


In [0]:
spark.conf.set(
    "fs.azure.account.key.capstonestore.dfs.core.windows.net",
    "Grl3eesbI5BdkB40Pqo2ETydRDB84D1e7WZ/OtRO4sw8P47jlp/fnaJ0Ws6u7qEdeAJDfgZntXd2+AStfg4LMw=="
)

In [0]:
spark.sql("USE CATALOG capstone")
spark.sql("USE SCHEMA bronze")

DataFrame[]

In [0]:
base_path = "abfss://dataset@capstonestore.dfs.core.windows.net/raw/"
df_adm = spark.read.option("header", True).option("inferSchema", True).option("mergeSchema", "true").csv(base_path + "admissions/")
df_pat = spark.read.option("header", True).option("inferSchema", True).option("mergeSchema", "true").csv(base_path + "patients/")
df_doc = spark.read.option("header", True).option("inferSchema", True).option("mergeSchema", "true").csv(base_path + "doctors/")
df_hosp = spark.read.option("header", True).option("inferSchema", True).option("mergeSchema", "true").csv(base_path + "hospitals/")
df_bill = spark.read.option("header", True).option("inferSchema", True).option("mergeSchema", "true").csv(base_path + "billing/")
df_lab = spark.read.option("header", True).option("inferSchema", True).option("mergeSchema", "true").csv(base_path + "labs/")

In [0]:
#filter with watermark

from pyspark.sql.functions import col

df_adm = df_adm.filter(col("admission_date") > watermark)

Understanding dataset


In [0]:
#Understand Structure
df_adm.printSchema()
df_adm.show()

root
 |-- admission_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- hospital_id: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- admission_type: string (nullable = true)
 |-- diagnosis_code: integer (nullable = true)
 |-- attending_doctor_id: integer (nullable = true)

+------------+----------+-----------+----------+--------------+--------------+--------------+--------------+-------------------+
|admission_id|patient_id|hospital_id|department|admission_date|discharge_date|admission_type|diagnosis_code|attending_doctor_id|
+------------+----------+-----------+----------+--------------+--------------+--------------+--------------+-------------------+
+------------+----------+-----------+----------+--------------+--------------+--------------+--------------+-------------------+



In [0]:
df_pat.printSchema()
df_pat.head()

root
 |-- patient_id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)
 |-- insurance_type: string (nullable = true)
 |-- chronic_condition_flag: integer (nullable = true)



Row(patient_id=10000, age=52.0, gender='Female', city='Pune', insurance_type='Private', chronic_condition_flag=0)

In [0]:
df_doc.printSchema()
df_doc.head()

root
 |-- doctor_id: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- experience_years: integer (nullable = true)



Row(doctor_id=5000, department='General', specialization='General', experience_years=8)

In [0]:
df_hosp.printSchema()
df_hosp.head()

root
 |-- hospital_id: integer (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- capacity_beds: integer (nullable = true)



Row(hospital_id=1, hospital_name='Hospital_0', city='Mumbai', capacity_beds=307)

In [0]:
df_lab.printSchema()
df_lab.head()

root
 |-- lab_test_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- test_name: string (nullable = true)
 |-- test_date: date (nullable = true)
 |-- test_result: double (nullable = true)
 |-- normal_range_flag: integer (nullable = true)
 |-- reported_at: timestamp (nullable = true)



Row(lab_test_id=900000, patient_id=72398, test_name='CT Scan', test_date=datetime.date(2022, 5, 11), test_result=53.46, normal_range_flag=0, reported_at=datetime.datetime(2022, 5, 12, 19, 0))

In [0]:
df_bill.printSchema()
df_bill.head()

root
 |-- bill_id: integer (nullable = true)
 |-- admission_id: integer (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- insurance_covered_amount: double (nullable = true)
 |-- patient_payable: double (nullable = true)
 |-- claim_status: string (nullable = true)



Row(bill_id=600000, admission_id=880234, total_charges=162776.95, insurance_covered_amount=95591.94, patient_payable=67185.01000000001, claim_status='Rejected')

In [0]:
spark.table("capstone.metadata.watermark").display()

table_name,last_loaded_date
admissions,2024-06-18
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01
admissions,2020-01-01


In [0]:
from pyspark.sql.functions import max

new_watermark = df_adm.agg(max("admission_date")).collect()[0][0]

if new_watermark is not None:
    spark.sql(f"""
    UPDATE capstone.metadata.watermark
    SET last_loaded_date = '{new_watermark}'
    WHERE table_name = 'admissions'
    """)
else:
    print("No new data → watermark not updated")

No new data → watermark not updated


Wriritng into bronze layer

In [0]:
df_adm.write.mode("overwrite").saveAsTable("capstone.bronze.admissions")
df_pat.write.mode("overwrite").saveAsTable("capstone.bronze.patients")
df_doc.write.mode("overwrite").saveAsTable("capstone.bronze.doctors")
df_hosp.write.mode("overwrite").saveAsTable("capstone.bronze.hospitals")
df_lab.write.mode("overwrite").saveAsTable("capstone.bronze.lab_results")
df_bill.write.mode("overwrite").saveAsTable("capstone.bronze.billing")